RNN - Erro dos pesos computados e usado somente durante a iteração

In [19]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE
FileName = ipynbname.name()

df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values

def PlotPredError(rtlo,w=9,h=3):
    s = len(rtlo.yWAPE)
    t = rtlo.t
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(w, h))
    axes = axes.flatten()
    ax1,ax2,ax3,ax4 = axes[0], axes[1], axes[2], axes[3]

    ax1.plot(t, rtlo.yR, color='black',label='Y-Real', linestyle='-')
    ax1.plot(t, rtlo.yP, color='blue',label='Y-Pred', linestyle='-')
    ax1.plot(t, rtlo.yL, color='blue', linestyle='--')
    ax1.plot(t, rtlo.yU, color='blue', linestyle='--')
    
    ax1.set_title('Y - Real x Prediction')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y', color='black')
    ax1.legend()

    ax2.plot(t, rtlo.eR, color='black',label='e-Real', linestyle='-')
    ax2.plot(t, rtlo.eP, color='blue',label='e-Pred', linestyle='-')
    ax2.set_title('Error - Real x Prediction')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Prediction Error', color='black') 
    ax2.legend()
    
    ax3.plot(t[-s:], rtlo.yWAPE, color='blue',label='WAPE', linestyle='-')
    ax3.set_title('Prediction WAPE')
    ax3.set_xlabel('X')
    ax3.set_ylabel('WAPE', color='black') 

    '''ax4.plot(t[-s:], rtlo.rWAPE, color='blue',label='WAPE', linestyle='-')
    ax4.set_title('RUL Prediction WAPE')
    ax4.set_xlabel('X')
    ax4.set_ylabel('WAPE', color='black') '''

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()

def PlotPredErrorPLY(rtlo, w=800, h=300):
    # s: tamanho do vetor WAPE (caso comece depois do início)
    t = rtlo.t
    
    # Criando o layout de 1 linha e 4 colunas
    fig = make_subplots(
        rows=1, cols=3, 
        shared_xaxes=True,
        subplot_titles=('Y - Real x Prediction', 'RUL - Real x Pred', 'RUL Prediction WAPE', 'Prediction WAPE', 'RUL Prediction WAPE')
    )

    # --- Subplot 1: Y Real x Pred (com Intervalos) ---
    fig.add_trace(go.Scatter(x=t, y=rtlo.yR, name='Y-Real', line=dict(color='black')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yP, name='Y-Pred', line=dict(color='blue')), row=1, col=1)
    # Intervalos (Dashed)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yL, name='y-Lower', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yU, name='y-Upper', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=rtlo.rulR, name='rul R', line=dict(color='black'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulP, name='rul P', line=dict(color='blue'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulL, name='rul L', line=dict(color='red'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulU, name='rul U', line=dict(color='green'), showlegend=True), row=1, col=2)

    fig.add_trace(go.Scatter(x=t[-len(rtlo.εR_hist):], y=rtlo.εR_hist, name='wMAPE', line=dict(color='black'), showlegend=True), row=1, col=3)
    #fig.add_trace(go.Scatter(x=t, y=rtlo.eR, name='erro R', line=dict(color='black')), row=1, col=3)
    #fig.add_trace(go.Scatter(x=t, y=rtlo.eP, name='erro P', line=dict(color='blue')), row=1, col=3)

    # Atualizando Layout e Eixos
    fig.update_layout(
        width=w, height=h,
        title_text=f"RTLO Model Performance Analysis",
        template='plotly_white',
        showlegend=True,
        margin=dict(l=40, r=40, t=80, b=40)
    )

    # Labels dos eixos (opcional, já que os títulos ajudam)
    fig.update_xaxes(title_text="Time / Index")
    fig.update_yaxes(title_text="Amplitude", col=1)
    fig.update_yaxes(title_text="Error", col=2)

    fig.show()
    
def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler

In [446]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,lr=1e-5):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None
        self.act = 'tanh'
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.08

        self.xPi = np.zeros(nI)
        self.hP, self.hU, self.hL = [np.zeros(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)

        self.rls = RLS_LogarithmicRegressor(0.95,1e7)
        
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.yP_hist = np.zeros(self.nI)
        self.eS = np.zeros(nI)
        self.eP = np.array([])
        self.eR = np.array([])

        self.εY, self.εM, self.εE, self.εR, self.ΣW = [0 for i in range(5)]
        self.εM_hist = np.array([])
        self.εR_hist = np.array([])

        self.μrWAPE = 0
        self.MPsum = 0

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0

        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,start=0,store=False,show=False):

        η1,η2,η3 = self.ηS      

        uS = self.wR @ self.hP + self.wI @ xP
        hP = self.hP*(1-1/self.τ) +Activation(uS,self.act)/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        self.pS = np.outer(dActivation(uS,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS,self.act),self.xPi)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.hP = hP
        self.xPi = xP

        if self.k>=start:
            W = self.k**2
            ΣW = self.ΣW + W
            ΔY = np.abs((yR-yP)/yR)
            ΔM = np.linalg.norm(ΔY,ord=2)
            ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-9))

            self.εY = ((self.εY*self.ΣW) + (W*ΔY[0]))/ΣW
            self.εM = ((self.εM*self.ΣW) + (W*ΔM))/ΣW
            self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
            self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[0])
            if self.k>=start:
                self.εM_hist = np.append(self.εM_hist,self.εM)
                self.εR_hist = np.append(self.εR_hist,self.εR)

        self.k = self.k+1
        self.t = np.append(self.t,self.k + self.nI)
        self.yP_hist = np.delete(np.append(self.yP_hist,yP[0]),0)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    '''def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        #print('yH:',self.yP_hist)
        #print('eS: ',self.eS)

        for i,y in enumerate(self.yP_hist):
            if y != 0:
                self.eS2[i] = self.rls.predict(np.abs(y))
        #print('eS2:',self.eS2)

        xP,xL,xU =x.copy(), (x-(self.ρ*self.eS2)).copy(),(x+(self.ρ*self.eS2)).copy()    

        #xP,xL,xU =x.copy(), x.copy()*0.999, x.copy()*1.00

        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
    
            yP = (wO@hP)
            yL = (wOp @ hL - wOn @ hU)
            yU = (wOp @ hU - wOn @ hL)

            #if show: print(yP)
            yP = yP[0]
            yL = yL[0]
            yU = yU[0]

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)
            PredVals = [yL,yP,yU]

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)'''
    
    def PredRulIntr2(self, x,lim=0.2,maxRul=110,store=False,show=False):
        
        xP,xL,xU =x.copy(), x.copy(), x.copy()
        k = 1
        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO
        hP = self.hP.copy() 
        ep = 1+self.ρ

        wRU, wRL = np.maximum(ep*wR, wR/ep), np.minimum(ep*wR, wR/ep)
        wIU, wIL = np.maximum(ep*wI, wI/ep), np.minimum(ep*wI, wI/ep)
        wOU, wOL = np.maximum(ep*wO, wO/ep), np.minimum(ep*wO, wO/ep) 
        hU ,  hL = np.maximum(ep*hP, hP/ep), np.minimum(ep*hP, hP/ep)

        '''ep=self.ρ

        wRU, wRL = np.maximum((1+ep)*wR, (1-ep)*wR), np.minimum((1+ep)*wR, (1-ep)*wR)
        wIU, wIL = np.maximum((1+ep)*wI, (1-ep)*wI), np.minimum((1+ep)*wI, (1-ep)*wI)
        wOU, wOL = np.maximum((1+ep)*wO, (1-ep)*wO), np.minimum((1+ep)*wO, (1-ep)*wO)
        hU, hL = np.maximum((1+ep)*hP, (1-ep)*hP), np.minimum((1+ep)*hP, (1-ep)*hP)'''

        while predict:
            uP = ( wR @ hP) + ( wI @ xP)
            uL = (wRL @ hL) + (wIL @ xL)
            uU = (wRU @ hU) + (wIU @ xU)

            #uU, uL = np.maximum(uU,uL), np.minimum(uU,uL)
            
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ

            hU,hL= hP,hP
            #hU, hL = np.maximum(hU,hL), np.minimum(hU,hL)

            yP = ( wO @ hP)
            yL = (wOL @ hL)
            yU = (wOU @ hU)
            yU, yL = np.maximum(yU,yL), np.minimum(yU,yL)

            
            PredVals = ([yL[0],yP[0],yU[0]])
            yL,yP,yU = PredVals

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
            k = k+1
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,lim=0.2,store=False):
        xP = x.copy()
        rulP=0
        predict = True
        hP = self.hP.copy()
        while predict:
            uP = (self.wR @ hP) + (self.wI @ xP)
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            yP = (self.wO @ hP)[0]
            xP = np.delete(np.append(xP,yP),0)
            if store:
                if rulP==0:
                    self.yP = np.append(self.yP,yP)

            if predict: rulP = rulP+1
            if yP < lim: predict = False
            if rulP >= 110:
                break

        self.rR=self.ref-self.k
        self.rP = rulP

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
            
    
    def UpdateRLS(self,yP,yR):
        eP = np.abs(self.rls.predict(np.abs(yP[0])))
        eR = np.abs(yP-yR)[0]
        self.rls.update(np.abs(yP[0]), eR)
        self.eR = np.append(self.eR,eR)
        self.eP = np.append(self.eP,eP)
        self.eS = np.append(self.eS,eP)
        self.eS = np.delete(self.eS,0)


#Optimize parameters for minimize error of degradation prediction

In [ ]:
rates = [1/(10**i) for i in range(1,7)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 8, 19) 
    nR = trial.suggest_int('nR', 30, 43) 
    nO = trial.suggest_int('nO', 1, 10) 
    N1 = trial.suggest_categorical('N1', rates[2:]) 
    N2 = trial.suggest_categorical('N2', rates[2:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 15)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.fit(X[i],Y[i],store=True)

        if i == 50:
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()

        if i % 20 == 0:  # Report every 10 time steps
            trial.report(rnn.εM, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
    #return rnn.εY
    return rnn.εM

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='auto'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=4000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

In [ ]:
rates = [1/(10**i) for i in range(1,8)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 2, 30) 
    nR = trial.suggest_int('nR', 1, 50) 
    nO = trial.suggest_int('nO', 1, 25) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 30)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    X,Y = X[:-1], Y[:-1]
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],store=True)
        rnn.fit(X[i],Y[i],start=50,store=True)

        if i % 25==0 and i >=50:
            if np.mean(rnn.rulP)<np.mean(rnn.rulR)*0.3 or np.mean(rnn.rulP)>np.mean(rnn.rulR)*1.12:
                raise TrialPruned()

        '''if i == 50:
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()
        if i % 20 == 0:  # Report every 10 time steps
            trial.report(rnn.εM, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()'''
            
    return rnn.εM, rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    directions=["minimize", "minimize"],
    sampler=SelSampler(mode='random'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=2500)

best_trials = study.best_trials

print(f"Encontrados {len(best_trials)} modelos na Fronteira de Pareto:")

for i, trial in enumerate(best_trials):
    print(f"Erro_M = {trial.values[0]:.6f}, Erro_R = {trial.values[1]:.6f}",
          f"Parâmetros: {trial.params}")
# Se você quiser apenas os parâmetros do PRIMEIRO modelo da fronteira para testar:
first_best_params = best_trials[0].params
params = list(trial.params.values())


[I 2026-04-17 12:07:10,552] A new study created in memory with name: no-name-446d8107-b7dd-4775-b8b3-7739a1807683
[I 2026-04-17 12:07:10,609] Trial 0 pruned. 
[I 2026-04-17 12:07:10,618] Trial 1 pruned. 
[I 2026-04-17 12:07:10,622] Trial 2 pruned. 
[I 2026-04-17 12:07:10,629] Trial 3 pruned. 
[I 2026-04-17 12:07:10,684] Trial 4 pruned. 
[I 2026-04-17 12:07:10,692] Trial 5 pruned. 
[I 2026-04-17 12:07:10,701] Trial 6 pruned. 
[I 2026-04-17 12:07:10,705] Trial 7 pruned. 
[I 2026-04-17 12:07:10,714] Trial 8 pruned. 
[I 2026-04-17 12:07:10,727] Trial 9 pruned. 
[I 2026-04-17 12:07:10,826] Trial 10 finished with values: [6.968007605159767, 2.549006249106573] and parameters: {'nI': 20, 'nR': 16, 'nO': 18, 'N1': 1e-07, 'N2': 0.01, 'N3': 1e-06, 'τ': 7}.
[I 2026-04-17 12:07:10,888] Trial 11 finished with values: [0.9171773306295692, 2.5943402479418896] and parameters: {'nI': 22, 'nR': 21, 'nO': 18, 'N1': 0.01, 'N2': 1e-06, 'N3': 0.1, 'τ': 25}.
[I 2026-04-17 12:07:11,001] Trial 12 finished with 

Encontrados 8 modelos na Fronteira de Pareto:
Erro_M = 2.085888, Erro_R = 0.074934 Parâmetros: {'nI': 17, 'nR': 49, 'nO': 3, 'N1': 1e-06, 'N2': 0.0001, 'N3': 1e-05, 'τ': 17}
Erro_M = 0.006675, Erro_R = 0.440473 Parâmetros: {'nI': 8, 'nR': 37, 'nO': 1, 'N1': 0.1, 'N2': 1e-07, 'N3': 1e-06, 'τ': 12}
Erro_M = 0.072930, Erro_R = 0.199125 Parâmetros: {'nI': 11, 'nR': 40, 'nO': 12, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ': 2}
Erro_M = 0.102567, Erro_R = 0.146271 Parâmetros: {'nI': 10, 'nR': 43, 'nO': 25, 'N1': 0.1, 'N2': 0.01, 'N3': 0.001, 'τ': 19}
Erro_M = 0.044032, Erro_R = 0.220112 Parâmetros: {'nI': 24, 'nR': 28, 'nO': 3, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 10}
Erro_M = 0.038431, Erro_R = 0.323773 Parâmetros: {'nI': 24, 'nR': 18, 'nO': 3, 'N1': 0.1, 'N2': 0.1, 'N3': 0.1, 'τ': 11}
Erro_M = 0.123427, Erro_R = 0.115910 Parâmetros: {'nI': 15, 'nR': 34, 'nO': 20, 'N1': 0.1, 'N2': 1e-05, 'N3': 1e-05, 'τ': 27}
Erro_M = 0.034552, Erro_R = 0.335804 Parâmetros: {'nI': 14, 'nR': 25, 'nO': 1, '

Erro_M = 0.132222, Erro_R = 0.145739 Parâmetros: {'nI': 18, 'nR': 41, 'nO': 16, 'N1': 0.1, 'N2': 0.001, 'N3': 0.001, 'τ': 21}\
Erro_M = 0.117815, Erro_R = 0.147648 Parâmetros: {'nI': 3, 'nR': 25, 'nO': 6, 'N1': 0.1, 'N2': 0.01, 'N3': 1e-06, 'τ': 18}\
Erro_M = 0.106091, Erro_R = 0.144622 Parâmetros: {'nI': 18, 'nR': 31, 'nO': 1, 'N1': 1e-07, 'N2': 0.001, 'N3': 1e-05, 'τ': 21}\
Erro_M = 0.063852, Erro_R = 0.121791 Parâmetros: {'nI': 15, 'nR': 34, 'nO': 3, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ': 25}\
Erro_M = 0.063259, Erro_R = 0.194122 Parâmetros: {'nI': 7, 'nR': 31, 'nO': 6, 'N1': 0.1, 'N2': 1e-05, 'N3': 1e-07, 'τ': 20}\
Erro_M = 0.049165, Erro_R = 0.169135 Parâmetros: {'nI': 22, 'nR': 48, 'nO': 6, 'N1': 0.1, 'N2': 0.001, 'N3': 0.0001, 'τ': 12}\





In [452]:
params= best_trials[-4].params
params = {'nI': 7, 'nR': 31, 'nO': 6, 'N1': 0.1, 'N2': 1e-05, 'N3': 1e-07, 'τ': 20}
params = list(params.values())
vec=[]

In [453]:
nI,nR,nO,N1,N2,N3,τ= params
X,Y = PrepareDataAhead(sig,n=nI,m=nO)
X,Y = X[:-1], Y[:-1]
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    #rnn.PredRul(x=X[i],store=True)
    rnn.PredRulIntr2(x=X[i],maxRul=110,store=True,show=False)
    rnn.fit(X[i],Y[i],store=True,start=40,show=False)
print(rnn.εM,rnn.εR)  
PlotPredErrorPLY(rnn,w=800,h=400)

0.054247355149469816 0.1878898505075924


In [408]:
PlotSeriesPLY(ySeries=vec)

In [ ]:
i=50
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.3
r_mU = np.mean(rnn.rulR[:i])*1.13
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

i=50
f=90
r_m = np.mean(rnn.rulR[i:f])
r_mL = np.mean(rnn.rulR[i:f])*0.3
r_mU = np.mean(rnn.rulR[i:f])*1.5
p_m = (np.mean(rnn.rulP[i:f]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

lower: 25.65 mid: 85.5 upper: 96.615
pred: 94.74
lower: 12.15 mid: 40.5 upper: 60.75
pred: 110.0


In [354]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ= params
ηS = [N1,N2,N3]
X,Y = PrepareDataAhead(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,ηS,τ)
rnn.ref = len(sig)-nI

i=0

In [358]:
#rnn.PredRul(x=X[i],store=True)
print('iter',i)
rnn.PredRulIntr2(x=X[i],store=True,show=True)
rnn.fit(X[i],Y[i],store=True,show=False)
i=i+1

iter 3
[-0.08467504 -0.22019617 -0.09249511  0.06758076 -0.13162155  0.07880601
 -0.2501543   0.21388642 -0.09186916 -0.146423   -0.21691833  0.03982975
  0.0525465  -0.09188978 -0.02597842  0.19339423  0.2006494   0.14826266
 -0.22694161 -0.08881666 -0.15404794 -0.0774944   0.10364182 -0.25478491
  0.20754114  0.09904901 -0.10291928  0.031353   -0.2138643   0.10705594
  0.02775517  0.22080556  0.10610808  0.20472202  0.06682817 -0.00468918
 -0.23644123 -0.27709793 -0.09889183 -0.14529573 -0.15335747 -0.22557529
  0.07375655]
[-0.06268424 -0.20075735 -0.07230302  0.08908711 -0.1100559   0.09797627
 -0.22831707  0.23464223 -0.07246881 -0.12532484 -0.19675578  0.06104215
  0.07687596 -0.07373077 -0.01006222  0.216947    0.22112495  0.16690248
 -0.20573463 -0.06942426 -0.13327741 -0.06440087  0.12583319 -0.23331603
  0.22867114  0.1230315  -0.08378739  0.05626652 -0.19309864  0.1358674
  0.04544649  0.24064836  0.13149472  0.22759046  0.08405777  0.0110672
 -0.21617602 -0.2565056  -0.0801